# dbt, the Tool That Turns SQL into a Tested Software Project

Notebook for **Chapter 7** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 Models, Sources, ref() and Materializations

In [ ]:
import re, sqlite3, graphlib
models={
 "stg_orders":"SELECT order_id, customer_id, amount/100.0 AS amount FROM raw_orders",
 "stg_customers":"SELECT id AS customer_id, UPPER(city) AS city FROM raw_customers",
 "fct_orders":"SELECT o.order_id, o.amount, c.city FROM {{ ref('stg_orders') }} o JOIN {{ ref('stg_customers') }} c USING(customer_id)",
 "mart_city_revenue":"SELECT city, SUM(amount) AS revenue FROM {{ ref('fct_orders') }} GROUP BY city",
}
db=sqlite3.connect(":memory:")
db.executescript("""CREATE TABLE raw_orders(order_id,customer_id,amount);INSERT INTO raw_orders VALUES(1,1,25000),(2,1,5000),(3,2,12000);
CREATE TABLE raw_customers(id,city);INSERT INTO raw_customers VALUES(1,'pune'),(2,'delhi');""")
deps={m:set(re.findall(r"ref\('(\w+)'\)",s)) for m,s in models.items()}
for m in graphlib.TopologicalSorter(deps).static_order():
    sql=re.sub(r"\{\{ ref\('(\w+)'\) \}\}",r"\1",models[m])
    db.execute(f"CREATE VIEW {m} AS {sql}"); print("built",m)
print(db.execute("SELECT * FROM mart_city_revenue ORDER BY city").fetchall())
print("test not_null(fct_orders.city) bad rows:",db.execute("SELECT COUNT(*) FROM fct_orders WHERE city IS NULL").fetchone()[0])

## 7.2 Incremental Models, Snapshots and Seeds

In [ ]:
import sqlite3
db=sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE raw_orders(order_id INT, customer_id INT, amount_paise INT, updated_at TEXT);
INSERT INTO raw_orders VALUES (1,1,25000,'2024-06-01'),(2,1,5000,'2024-06-01'),(3,2,12000,'2024-06-02');
CREATE TABLE fct_orders(order_id INT PRIMARY KEY, customer_id INT, amount REAL, updated_at TEXT);
""")
def run_incremental(full_refresh=False):
    if full_refresh: db.execute("DELETE FROM fct_orders")
    wm=db.execute("SELECT COALESCE(MAX(updated_at),'1900-01-01') FROM fct_orders").fetchone()[0]
    db.execute("""INSERT OR REPLACE INTO fct_orders
      SELECT order_id, customer_id, amount_paise/100.0, updated_at FROM raw_orders WHERE updated_at > ?""",(wm,))
    return wm
print("run 1 watermark:",run_incremental(), "rows:",db.execute("SELECT COUNT(*) FROM fct_orders").fetchone()[0])
db.execute("INSERT INTO raw_orders VALUES (4,3,8000,'2024-06-03')")
db.execute("UPDATE raw_orders SET amount_paise=30000, updated_at='2024-06-03' WHERE order_id=1")
print("run 2 watermark:",run_incremental(), "rows:",db.execute("SELECT COUNT(*) FROM fct_orders").fetchone()[0])
print(db.execute("SELECT * FROM fct_orders ORDER BY order_id").fetchall())
print("run 3 (nothing new) watermark:",run_incremental(),"rows:",db.execute("SELECT COUNT(*) FROM fct_orders").fetchone()[0])
db.execute("INSERT INTO raw_orders VALUES (5,2,9900,'2024-06-01')")
run_incremental(); print("late row 5 loaded?",db.execute("SELECT COUNT(*) FROM fct_orders WHERE order_id=5").fetchone()[0]==1)
run_incremental(full_refresh=True); print("after full refresh, loaded?",db.execute("SELECT COUNT(*) FROM fct_orders WHERE order_id=5").fetchone()[0]==1)

## 7.3 Tests, Macros, Packages, Docs and CI

## 7.4 Incremental Strategies Compared on One Dataset

In [ ]:
import sqlite3

def run(strategy, lookback_days=0):
    db = sqlite3.connect(":memory:")
    db.executescript("""
    CREATE TABLE src(order_id INT, amount INT, updated_at TEXT);
    CREATE TABLE tgt(order_id INT, amount INT, updated_at TEXT);
    CREATE UNIQUE INDEX u ON tgt(order_id);
    """ if strategy == "merge" else """
    CREATE TABLE src(order_id INT, amount INT, updated_at TEXT);
    CREATE TABLE tgt(order_id INT, amount INT, updated_at TEXT);
    """)
    def incremental_run():
        db.execute("DROP TABLE IF EXISTS batch")
        db.execute(f"""CREATE TEMP TABLE batch AS SELECT * FROM src WHERE updated_at >
                      date(COALESCE((SELECT MAX(updated_at) FROM tgt), '1900-01-01'), '-{lookback_days} day')""")
        if strategy == "append":
            db.execute("INSERT INTO tgt SELECT * FROM batch")
        elif strategy == "delete+insert":
            db.execute("DELETE FROM tgt WHERE order_id IN (SELECT order_id FROM batch)")
            db.execute("INSERT INTO tgt SELECT * FROM batch")
        else:  # merge (an upsert on the unique key)
            db.execute("""INSERT INTO tgt SELECT * FROM batch WHERE true
                          ON CONFLICT(order_id) DO UPDATE
                          SET amount = excluded.amount, updated_at = excluded.updated_at""")
    db.executemany("INSERT INTO src VALUES (?,?,?)", [(1, 100, "2024-06-01"), (2, 200, "2024-06-01"), (3, 300, "2024-06-02")])
    incremental_run()                                                   # run 1
    db.execute("UPDATE src SET amount = 250, updated_at = '2024-06-03' WHERE order_id = 2")
    db.executemany("INSERT INTO src VALUES (?,?,?)", [(4, 400, "2024-06-03"), (5, 500, "2024-06-03"),
                                                       (6, 600, "2024-06-02")])   # order 6 landed late
    incremental_run()                                                   # run 2
    return db.execute("SELECT order_id, amount FROM tgt ORDER BY order_id, amount").fetchall()

for s in ["append", "delete+insert", "merge"]:
    r = run(s)
    print(f"{s:14s} lookback 0: {len(r)} rows", r)
r = run("merge", 1);  print(f"{'merge':14s} lookback 1: {len(r)} rows", r)
r = run("append", 1); print(f"{'append':14s} lookback 1: {len(r)} rows", r)

append         lookback 0: 6 rows [(1, 100), (2, 200), (2, 250), (3, 300), (4, 400), (5, 500)]
delete+insert  lookback 0: 5 rows [(1, 100), (2, 250), (3, 300), (4, 400), (5, 500)]
merge          lookback 0: 5 rows [(1, 100), (2, 250), (3, 300), (4, 400), (5, 500)]
merge          lookback 1: 6 rows [(1, 100), (2, 250), (3, 300), (4, 400), (5, 500), (6, 600)]
append         lookback 1: 8 rows [(1, 100), (2, 200), (2, 250), (3, 300), (3, 300), (4, 400), (5, 500), (6, 600)]


## 7.5 A Snapshot Walked Through, Step by Step

Section 7.2 gave the idea. Now watch a snapshot work, run by run, for both strategies, on one source table of customers. Strategy `timestamp` uses the source's `updated_at` as the clock and notices a change when that stamp is newer than the stored row. Strategy `check` notices a change only when a watched column (here `tier`) differs, and uses the day the snapshot ran as the clock. Before you run it: on run 3 we only *touch* customer 2 (new `updated_at`, same tier). Which strategy creates a new history row?

In [ ]:
import sqlite3

def make(strategy):
    db = sqlite3.connect(":memory:")
    db.executescript("""
    CREATE TABLE src(customer_id INT, tier TEXT, updated_at TEXT);
    CREATE TABLE snap(customer_id INT, tier TEXT, valid_from TEXT, valid_to TEXT);
    """)
    def snapshot(run_date):
        for cid, tier, upd in db.execute("SELECT * FROM src").fetchall():
            cur = db.execute("SELECT tier, valid_from FROM snap WHERE customer_id=? AND valid_to IS NULL", (cid,)).fetchone()
            stamp = upd if strategy == "timestamp" else run_date
            changed = cur is None or (upd > cur[1] if strategy == "timestamp" else tier != cur[0])
            if cur is not None and changed:
                db.execute("UPDATE snap SET valid_to=? WHERE customer_id=? AND valid_to IS NULL", (stamp, cid))
            if changed:
                db.execute("INSERT INTO snap VALUES (?,?,?,NULL)", (cid, tier, stamp))
    return db, snapshot

def show(db, title):
    print(title)
    for r in db.execute("SELECT * FROM snap ORDER BY customer_id, valid_from"): print("  ", r)

for strategy in ["timestamp", "check"]:
    db, snapshot = make(strategy)
    db.executemany("INSERT INTO src VALUES (?,?,?)", [(1, "silver", "2024-05-20"), (2, "gold", "2024-05-25")])
    snapshot("2024-06-01"); show(db, f"[{strategy}] after run 1 (06-01)")
    db.execute("UPDATE src SET tier='gold', updated_at='2024-06-09' WHERE customer_id=1")
    db.execute("INSERT INTO src VALUES (3,'silver','2024-06-08')")
    snapshot("2024-06-10"); show(db, f"[{strategy}] after run 2 (06-10)")
    db.execute("UPDATE src SET updated_at='2024-06-14' WHERE customer_id=2")      # touched, tier unchanged
    snapshot("2024-06-15"); show(db, f"[{strategy}] after run 3 (06-15), customer 2 touched, tier unchanged")

[timestamp] after run 1 (06-01)
   (1, 'silver', '2024-05-20', None)
   (2, 'gold', '2024-05-25', None)
[timestamp] after run 2 (06-10)
   (1, 'silver', '2024-05-20', '2024-06-09')
   (1, 'gold', '2024-06-09', None)
   (2, 'gold', '2024-05-25', None)
   (3, 'silver', '2024-06-08', None)
[timestamp] after run 3 (06-15), customer 2 touched, tier unchanged
   (1, 'silver', '2024-05-20', '2024-06-09')
   (1, 'gold', '2024-06-09', None)
   (2, 'gold', '2024-05-25', '2024-06-14')
   (2, 'gold', '2024-06-14', None)
   (3, 'silver', '2024-06-08', None)
[check] after run 1 (06-01)
   (1, 'silver', '2024-06-01', None)
   (2, 'gold', '2024-06-01', None)
[check] after run 2 (06-10)
   (1, 'silver', '2024-06-01', '2024-06-10')
   (1, 'gold', '2024-06-10', None)
   (2, 'gold', '2024-06-01', None)
   (3, 'silver', '2024-06-10', None)
[check] after run 3 (06-15), customer 2 touched, tier unchanged
   (1, 'silver', '2024-06-01', '2024-06-10')
   (1, 'gold', '2024-06-10', None)
   (2, 'gold', '2024-06-0

## 7.6 Macros, Custom Tests and Selecting What to Build

In [ ]:
import sqlite3
from jinja2 import Template

tpl = Template("""SELECT order_id,
{%- for m in methods %}
       SUM(CASE WHEN method = '{{ m }}' THEN amount ELSE 0 END) AS {{ m }}_amount{{ "," if not loop.last }}
{%- endfor %}
FROM payments
{%- if target == "dev" %}
WHERE order_id <= 2   -- dev: only a small sample
{%- endif %}
GROUP BY order_id ORDER BY order_id""")

db = sqlite3.connect(":memory:")
db.executescript("""CREATE TABLE payments(order_id INT, method TEXT, amount INT);
INSERT INTO payments VALUES (1,'card',100),(1,'upi',50),(2,'upi',200),(3,'card',300),(3,'cod',75);""")
for target in ["dev", "prod"]:
    sql = tpl.render(methods=["card", "upi", "cod"], target=target)
    print(f"--- target={target} ---")
    if target == "dev": print(sql)
    print(db.execute(sql).fetchall())

--- target=dev ---
SELECT order_id,
       SUM(CASE WHEN method = 'card' THEN amount ELSE 0 END) AS card_amount,
       SUM(CASE WHEN method = 'upi' THEN amount ELSE 0 END) AS upi_amount,
       SUM(CASE WHEN method = 'cod' THEN amount ELSE 0 END) AS cod_amount
FROM payments
WHERE order_id <= 2   -- dev: only a small sample
GROUP BY order_id ORDER BY order_id
[(1, 100, 50, 0), (2, 0, 200, 0)]
--- target=prod ---
[(1, 100, 50, 0), (2, 0, 200, 0), (3, 300, 0, 75)]


In [ ]:
import sqlite3
from jinja2 import Template
test_sql = Template("select * from {{ model }} where {{ column_name }} <= 0 or {{ column_name }} is null")
db = sqlite3.connect(":memory:")
db.executescript("CREATE TABLE stg_orders(order_id INT, amount INT); INSERT INTO stg_orders VALUES (1,100),(2,0),(3,-5),(4,NULL),(5,40);")
sql = test_sql.render(model="stg_orders", column_name="amount")
print(sql)
failures = db.execute(sql).fetchall()
print("failing rows:", failures)
print("test result:", "PASS" if not failures else f"FAIL ({len(failures)} rows)")

select * from stg_orders where amount <= 0 or amount is null
failing rows: [(2, 0), (3, -5), (4, None)]
test result: FAIL (3 rows)


In [ ]:
parents = {"stg_orders": [], "stg_customers": [], "fct_orders": ["stg_orders", "stg_customers"],
           "mart_revenue_by_city": ["fct_orders"], "mart_customer_ltv": ["fct_orders"]}

def descendants(node):
    out = set()
    for m, ps in parents.items():
        if node in ps:
            out |= {m} | descendants(m)
    return out

def ancestors(node):
    out = set()
    for p in parents[node]:
        out |= {p} | ancestors(p)
    return out

modified = {"stg_customers"}
plus = set(modified)
for m in modified: plus |= descendants(m)
print("state:modified+      ->", sorted(plus))
print("fct_orders+          ->", sorted({"fct_orders"} | descendants("fct_orders")))
print("+mart_customer_ltv   ->", sorted({"mart_customer_ltv"} | ancestors("mart_customer_ltv")))
print("not rebuilt in CI    ->", sorted(set(parents) - plus))

state:modified+      -> ['fct_orders', 'mart_customer_ltv', 'mart_revenue_by_city', 'stg_customers']
fct_orders+          -> ['fct_orders', 'mart_customer_ltv', 'mart_revenue_by_city']
+mart_customer_ltv   -> ['fct_orders', 'mart_customer_ltv', 'stg_customers', 'stg_orders']
not rebuilt in CI    -> ['stg_orders']


## Chapter recap

## Practice set

**1.** (easy) What does `dbt run --select fct_orders` build? **Answer:** only `fct_orders` (no `+`, so no upstream and no downstream).

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch07.html